# 🔧 Predictive Maintenance — Exploratory Data Analysis

**Objective:** Understand the sensor dataset, identify patterns, detect class imbalance, and guide feature engineering decisions.

**Contents**
1. Setup & Data Loading
2. Basic Statistics
3. Missing Value Analysis
4. Class Imbalance
5. Feature Distributions
6. Correlation Analysis
7. Failure Pattern Visualisation
8. Outlier Detection
9. Feature vs Target Relationships
10. Key Findings Summary

## 1. Setup & Data Loading

In [ ]:
import sys, os
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

# Project modules
from src.data_loader import load_data, generate_synthetic_dataset
from src.utils import describe_dataframe, class_balance, check_memory

# Plot style
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_theme(style='whitegrid', palette='muted')

# Colour palette
COLORS = {'no_failure': '#2ca02c', 'failure': '#d62728'}
print('Libraries loaded ✅')

In [ ]:
RAW_PATH = '../data/raw/sensor_data.csv'
os.makedirs('../data/raw', exist_ok=True)

if not Path(RAW_PATH).exists():
    print('No CSV found — generating synthetic dataset (5,000 samples).')
    df_raw = generate_synthetic_dataset(n_samples=5000)
    df_raw.to_csv(RAW_PATH, index=False)

df = load_data(RAW_PATH)
print(f'Dataset shape: {df.shape}')
df.head()

## 2. Basic Statistics

In [ ]:
print(f'Memory usage: {check_memory(df)}')
print(f'Shape:        {df.shape}')
print(f'Columns:      {list(df.columns)}')
print()
describe_dataframe(df)

In [ ]:
print('Data types:')
print(df.dtypes)
print()
print('Missing values:')
print(df.isnull().sum())

## 3. Class Imbalance Analysis

In [ ]:
balance = class_balance(df, 'failure')
print('Class balance:')
print(balance)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
labels = ['No Failure', 'Failure']
colours = [COLORS['no_failure'], COLORS['failure']]
axes[0].bar(labels, balance['count'], color=colours, edgecolor='k', linewidth=0.5)
for i, (count, pct) in enumerate(zip(balance['count'], balance['pct'])):
    axes[0].text(i, count + 20, f'{count}\n({pct}%)', ha='center', fontsize=11)
axes[0].set(title='Class Distribution', ylabel='Count')
axes[0].set_ylim(0, balance['count'].max() * 1.2)

# Pie
axes[1].pie(balance['count'], labels=labels, colors=colours, autopct='%1.1f%%',
            startangle=140, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Failure Proportion')

plt.tight_layout()
plt.savefig('../reports/figures/class_balance.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Feature Distributions

In [ ]:
numeric_features = ['temperature', 'vibration', 'pressure', 'voltage', 'runtime_hours',
                    'humidity', 'rotational_speed', 'torque', 'wear_level']
available = [c for c in numeric_features if c in df.columns]

n_cols = 3
n_rows = (len(available) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3.5))
axes = axes.flatten()

for i, col in enumerate(available):
    for label, colour in [(0, COLORS['no_failure']), (1, COLORS['failure'])]:
        axes[i].hist(df[df['failure'] == label][col], bins=40, alpha=0.6,
                     color=colour, label=f'{"No Failure" if label==0 else "Failure"}')
    axes[i].set(title=col.replace('_', ' ').title(), xlabel=col, ylabel='Count')
    axes[i].legend(fontsize=8)

# Hide unused
for j in range(len(available), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions by Failure Class', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Boxplots — Feature vs Failure

In [ ]:
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3.5))
axes = axes.flatten()

for i, col in enumerate(available):
    data_groups = [df[df['failure'] == 0][col].dropna(),
                   df[df['failure'] == 1][col].dropna()]
    bp = axes[i].boxplot(data_groups, patch_artist=True,
                         medianprops=dict(color='black', linewidth=2))
    bp['boxes'][0].set_facecolor(COLORS['no_failure'])
    bp['boxes'][0].set_alpha(0.6)
    if len(bp['boxes']) > 1:
        bp['boxes'][1].set_facecolor(COLORS['failure'])
        bp['boxes'][1].set_alpha(0.6)
    axes[i].set(title=col.replace('_', ' ').title(),
                xticklabels=['No Failure', 'Failure'])

for j in range(len(available), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Boxplots: Feature Values by Failure Class', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Correlation Heatmap

In [ ]:
corr_cols = [c for c in available + ['failure'] if c in df.columns]
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8}
)
ax.set_title('Correlation Matrix (lower triangle)', fontsize=14)
plt.tight_layout()
plt.savefig('../reports/figures/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Correlation with target
print('\nCorrelation with failure label:')
print(corr['failure'].drop('failure').sort_values(ascending=False).to_string())

## 7. Failure Pattern — Runtime Trends

In [ ]:
if 'runtime_hours' in df.columns:
    df['runtime_bin'] = pd.cut(df['runtime_hours'], bins=20)
    failure_by_runtime = df.groupby('runtime_bin', observed=True)['failure'].agg(['mean', 'count'])
    failure_by_runtime.columns = ['failure_rate', 'count']

    fig, ax1 = plt.subplots(figsize=(13, 5))
    x = range(len(failure_by_runtime))
    bars = ax1.bar(x, failure_by_runtime['failure_rate'], color='#d62728', alpha=0.65, label='Failure Rate')
    ax1.set_ylabel('Failure Rate', color='#d62728')
    ax1.set_xticks(x)
    ax1.set_xticklabels([str(i) for i in failure_by_runtime.index], rotation=45, ha='right', fontsize=8)
    ax1.set_title('Failure Rate vs. Runtime Hours')

    ax2 = ax1.twinx()
    ax2.plot(x, failure_by_runtime['count'], 'b-o', lw=1.5, ms=4, label='Sample Count')
    ax2.set_ylabel('Sample Count', color='blue')

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

    plt.tight_layout()
    plt.savefig('../reports/figures/failure_vs_runtime.png', dpi=150, bbox_inches='tight')
    plt.show()

## 8. Outlier Detection — Modified Z-Score

In [ ]:
outlier_summary = []
for col in available:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_summary.append({'feature': col, 'n_outliers': n_outliers,
                             'pct': round(n_outliers / len(df) * 100, 2),
                             'lower_fence': round(lower, 3), 'upper_fence': round(upper, 3)})

outlier_df = pd.DataFrame(outlier_summary)
print('Outlier summary (IQR method):')
print(outlier_df.to_string(index=False))

## 9. Feature vs. Target Scatter Plots

In [ ]:
key_pairs = [('runtime_hours', 'temperature'), ('vibration', 'temperature'),
             ('wear_level', 'vibration'), ('pressure', 'voltage')]

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes = axes.flatten()

for idx, (x_col, y_col) in enumerate(key_pairs):
    if x_col not in df.columns or y_col not in df.columns:
        continue
    for label, colour, marker in [(0, COLORS['no_failure'], 'o'), (1, COLORS['failure'], 'X')]:
        subset = df[df['failure'] == label]
        axes[idx].scatter(subset[x_col], subset[y_col], c=colour, alpha=0.25,
                          s=10, marker=marker, label=f'{"No Failure" if label==0 else "Failure"}')
    axes[idx].set(xlabel=x_col.replace('_', ' ').title(),
                  ylabel=y_col.replace('_', ' ').title(),
                  title=f'{x_col} vs {y_col}')
    axes[idx].legend(markerscale=2, fontsize=9)

plt.suptitle('Scatter Plots: Key Feature Pairs by Failure Class', fontsize=14)
plt.tight_layout()
plt.savefig('../reports/figures/scatter_pairs.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. RUL Distribution (if available)

In [ ]:
if 'rul' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    axes[0].hist(df['rul'], bins=50, color='steelblue', edgecolor='white', linewidth=0.3)
    axes[0].set(title='RUL Distribution (all samples)', xlabel='RUL (hours)', ylabel='Count')

    axes[1].hist(df[df['failure'] == 0]['rul'], bins=40, alpha=0.7, color=COLORS['no_failure'],
                 label='No Failure', edgecolor='white', linewidth=0.3)
    axes[1].hist(df[df['failure'] == 1]['rul'], bins=40, alpha=0.7, color=COLORS['failure'],
                 label='Failure', edgecolor='white', linewidth=0.3)
    axes[1].set(title='RUL Distribution by Failure Class', xlabel='RUL (hours)', ylabel='Count')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig('../reports/figures/rul_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('RUL statistics:')
    print(df.groupby('failure')['rul'].describe().round(2))
else:
    print('RUL column not found in dataset.')

## 11. Key Findings Summary

In [ ]:
print('=' * 60)
print('KEY FINDINGS SUMMARY')
print('=' * 60)
print(f'Dataset size:      {len(df):,} samples')
print(f'Features:          {len(df.columns) - 2} sensor inputs')
failure_rate = df['failure'].mean()
print(f'Failure rate:      {failure_rate:.1%}  ({df["failure"].sum()} failures)')
print(f'Imbalance ratio:   1:{int((1-failure_rate)/failure_rate):.0f}')
print()
print('Top features correlated with failure:')
corr_target = df[available + ['failure']].corr()['failure'].drop('failure').sort_values(key=abs, ascending=False)
for feat, val in corr_target.head(5).items():
    print(f'  {feat:<20} r = {val:.3f}')
print()
print('Recommendations:')
print('  ✅ Use SMOTE to handle class imbalance')
print('  ✅ Engineer runtime-based and interaction features')
print('  ✅ Scale features before SVM / Logistic Regression')
print('  ✅ Optimise classification threshold for high recall')
print('  ✅ Use tree-based models as primary candidates')